# Use Ragas to evaluate RAG pipeline

Ragas 是一个用于评估 RAG 组件的开源项目。包含论文、代码、文档和入门博客。

<img src="./ragas_eval_image.png">

**请注意，RAGAS 可能会消耗大量 OpenAI API token。**

请仔细阅读本笔记本，并注意您希望评估的问题数量和指标。

## 1. 准备Ragas环境和真实数据

Read questions and ground truth answers into a pandas dataframe.

> Note: 用 ''' 将每个上下文字符串括起来，以避免内部引号引起的问题。
>
> Note: 用逗号分隔每个上下文字符串.


In [1]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

cwd=os.getcwd()
relative_path='/data/blog_eval_answers.csv'
file_path=cwd+relative_path

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'

os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

os.environ['TORCH_HOME'] = CUSTOM_CACHE

# Read ground truth answers from file
eval_df=pd.read_csv(file_path,header=0,skip_blank_lines=True)
display(eval_df)

,Question,ground_truth_answer,recursive_context_512_k_2,html_context_512_k_2,parent_context_1536_k1,semantic_context_k_1,semantic_context_k_2_summary,parent_context_1536_k1_text-embedding-3-small,Custom_RAG_answer,llama3_ollama_answer,llama3_anyscale_answer,llama3_octoai_answer,llama3_groq_answer,mixtral_8x7b_anyscale_answer
0,What do the parameters for HNSW mean?,"* M: maximum degree, or number of connections ...","node closest to the target in this layer, and ...","layer, finds the node closest to the target in...",Parameter Description Range Default value npro...,Index building parameters Parameter Descriptio...,* `M`: The maximum number of outgoing connecti...,Parameter Description Range Default value npro...,The parameters for HNSW (Hierarchical Navigabl...,* `M`: Maximum number of outgoing connections ...,* M: Maximum number of outgoing connections in...,* `M`: Maximum number of outgoing connections ...,* M: Maximum number of outgoing connections in...,"The parameters for HNSW, a graph-based indexin..."
1,What are good default values for HNSW paramete...,"M=16, efConstruction=32, ef=32",Select your Milvus distribution first. Index b...,you can set the top-K up to 8192 for any searc...,Select your Milvus distribution first. Index b...,Index building parameters Parameter Descriptio...,* `nlist`: This parameter controls the number ...,Parameter Description Range Default value npro...,* M=48 * efConstruction=200,"* For `nprobe`, a reasonable default value is ...",* nprobe: around 10-20 * reorder_k: top_k (de...,* nprobe: [1-25] * reorder_k: top_k (not spec...,* nprobe: 16 * reorder_k: 128 * M: 128 * ef...,NaN
2,What does nlist vs nprobe mean in ivf_flat?,# nlist: controls how the vector data is part...,FAQ What is the difference between FLAT index ...,performance can be improved with minimal impac...,FAQ What is the difference between FLAT index ...,See Supported Metrics. IVF_FLAT IVF_FLAT divid...,**nlist (Number of List)**:\nThis parameter de...,1] FAQ What is the difference between FLAT ind...,- nlist refers to the number of clusters into ...,- `nlist` refers to the number of clusters (or...,- `nlist` refers to the number of clusters tha...,- `nlist` refers to the number of clusters tha...,- `nlist` refers to the number of clusters to ...,- `nlist` refers to the number of clusters in ...
3,What is the default AUTOINDEX index and vector...,Index type = HNSW and distance metric=IP Inner...,"True, and auto_id is enabled for the primary k...","is set to True, and auto_id is enabled for the...","vector in the data to be inserted, are treated...","For a detailed explanation of the schema, refe...","According to the provided text, when creating ...","vector in the data to be inserted, are treated...",The default AUTOINDEX index in Milvus is IVF_S...,The default `AUTOINDEX` index type uses L2 (Eu...,The default distance metric for vector fields ...,Milvus uses COSINE as the default distance met...,The default distance metric for vector fields ...,The default distance metric for vector fields ...


在下方的单元格中：选择要评估的 LLM 回答列（LLM_TO_EVALUATE，写入 Custom_RAG_answer），并选择要评估的检索上下文列（CONTEXT_TO_EVALUATE，写入 Custom_RAG_context）。

In [2]:
LLM_TO_EVALUATE = 'llama3_ollama_answer'

# 选择要评估的 Milvus 检索上下文列。
# blog_eval_answers.csv 中可用的列：
#   recursive_context_512_k_2, html_context_512_k_2, parent_context_1536_k1,
#   semantic_context_k_1, semantic_context_k_2_summary,
#   parent_context_1536_k1_text-embedding-3-small
CONTEXT_TO_EVALUATE = 'recursive_context_512_k_2'

temp_df=eval_df.copy()
if LLM_TO_EVALUATE != 'Custom_RAG_answer':
    temp_df['Custom_RAG_answer']=temp_df[LLM_TO_EVALUATE]
temp_df['Custom_RAG_context']=temp_df[CONTEXT_TO_EVALUATE]

display(temp_df.head())

,Question,ground_truth_answer,recursive_context_512_k_2,html_context_512_k_2,parent_context_1536_k1,semantic_context_k_1,semantic_context_k_2_summary,parent_context_1536_k1_text-embedding-3-small,Custom_RAG_answer,llama3_ollama_answer,llama3_anyscale_answer,llama3_octoai_answer,llama3_groq_answer,mixtral_8x7b_anyscale_answer,Custom_RAG_context
0,What do the parameters for HNSW mean?,"* M: maximum degree, or number of connections ...","node closest to the target in this layer, and ...","layer, finds the node closest to the target in...",Parameter Description Range Default value npro...,Index building parameters Parameter Descriptio...,* `M`: The maximum number of outgoing connecti...,Parameter Description Range Default value npro...,* `M`: Maximum number of outgoing connections ...,* `M`: Maximum number of outgoing connections ...,* M: Maximum number of outgoing connections in...,* `M`: Maximum number of outgoing connections ...,* M: Maximum number of outgoing connections in...,"The parameters for HNSW, a graph-based indexin...","node closest to the target in this layer, and ..."
1,What are good default values for HNSW paramete...,"M=16, efConstruction=32, ef=32",Select your Milvus distribution first. Index b...,you can set the top-K up to 8192 for any searc...,Select your Milvus distribution first. Index b...,Index building parameters Parameter Descriptio...,* `nlist`: This parameter controls the number ...,Parameter Description Range Default value npro...,"* For `nprobe`, a reasonable default value is ...","* For `nprobe`, a reasonable default value is ...",* nprobe: around 10-20 * reorder_k: top_k (de...,* nprobe: [1-25] * reorder_k: top_k (not spec...,* nprobe: 16 * reorder_k: 128 * M: 128 * ef...,NaN,Select your Milvus distribution first. Index b...
2,What does nlist vs nprobe mean in ivf_flat?,# nlist: controls how the vector data is part...,FAQ What is the difference between FLAT index ...,performance can be improved with minimal impac...,FAQ What is the difference between FLAT index ...,See Supported Metrics. IVF_FLAT IVF_FLAT divid...,**nlist (Number of List)**:\nThis parameter de...,1] FAQ What is the difference between FLAT ind...,- `nlist` refers to the number of clusters (or...,- `nlist` refers to the number of clusters (or...,- `nlist` refers to the number of clusters tha...,- `nlist` refers to the number of clusters tha...,- `nlist` refers to the number of clusters to ...,- `nlist` refers to the number of clusters in ...,FAQ What is the difference between FLAT index ...
3,What is the default AUTOINDEX index and vector...,Index type = HNSW and distance metric=IP Inner...,"True, and auto_id is enabled for the primary k...","is set to True, and auto_id is enabled for the...","vector in the data to be inserted, are treated...","For a detailed explanation of the schema, refe...","According to the provided text, when creating ...","vector in the data to be inserted, are treated...",The default `AUTOINDEX` index type uses L2 (Eu...,The default `AUTOINDEX` index type uses L2 (Eu...,The default distance metric for vector fields ...,Milvus uses COSINE as the default distance met...,The default distance metric for vector fields ...,The default distance metric for vector fields ...,"True, and auto_id is enabled for the primary k..."


In [3]:
# Ragas default uses HuggingFace Datasets.
from datasets import Dataset

def assemble_ragas_dataset(input_df):
    """从输入的 pandas 数据框中构建 RAGAS HuggingFace 数据集。"""

    # Assemble Ragas lists: questions, ground_truth_answers, retrieval_contexts, and RAG answers.
    question_list,truth_list,context_list=[],[],[]

    #　Get all the question
    question_list=input_df.Question.to_list()

    # Get all the ground truth answers
    truth_list=input_df.ground_truth_answer.to_list()

    # Get all the Milvus Retrieval Contexts as list[list[str]]
    if 'Custom_RAG_context' not in input_df.columns:
        raise KeyError("Missing 'Custom_RAG_context' column. Run the cell above (LLM_TO_EVALUATE) first, "
                       "then pass temp_df, not eval_df.")
    context_list=input_df.Custom_RAG_context.to_list()
    context_list=[[context] for context in context_list]

    # Get all the RAG answers based on contexts
    rag_answer_list=input_df.Custom_RAG_answer.to_list()

    # Create a HuggingFace Dataset from the ground truth lists
    rags_ds=Dataset.from_dict({
        "question":question_list,
        "contexts":context_list,
        "answer":rag_answer_list,
        "ground_truth":truth_list,
    })

    return rags_ds

In [4]:
# 从真实标注列表中创建一个 Ragas HuggingFace 数据集。
ragas_input_ds=assemble_ragas_dataset(temp_df)
display(ragas_input_ds)

Dataset({
    features: ['question', 'contexts', 'answer', 'ground_truth'],
    num_rows: 4
})

In [5]:
# 调试时检查所有数据
ragas_input_df=ragas_input_ds.to_pandas()
display(ragas_input_df.head())

,question,contexts,answer,ground_truth
0,What do the parameters for HNSW mean?,"[node closest to the target in this layer, and...",* `M`: Maximum number of outgoing connections ...,"* M: maximum degree, or number of connections ..."
1,What are good default values for HNSW paramete...,[Select your Milvus distribution first. Index ...,"* For `nprobe`, a reasonable default value is ...","M=16, efConstruction=32, ef=32"
2,What does nlist vs nprobe mean in ivf_flat?,[FAQ What is the difference between FLAT index...,- `nlist` refers to the number of clusters (or...,# nlist: controls how the vector data is part...
3,What is the default AUTOINDEX index and vector...,"[True, and auto_id is enabled for the primary ...",The default `AUTOINDEX` index type uses L2 (Eu...,Index type = HNSW and distance metric=IP Inner...


## 2. Start Ragas Evaluation with custom Evaluation LLM

Ragas 默认使用的模型是 deepseek-v4-flash。

请注意，每次提问和评估都会消耗大量 token。请留意您的令牌使用情况。

In [6]:
# ---- vertexai shim：langchain-community 0.4.2 移除了 chat_models.vertexai，
# 而 ragas 0.4.x 仍会 import ChatVertexAI（仅用于 isinstance 检查，不会被实例化）。
# 在 import ragas 之前注入 stub 模块，避免改动已安装的包。
import sys, types

_vmod = types.ModuleType('langchain_community.chat_models.vertexai')
class ChatVertexAI:  # placeholder
    pass
_vmod.ChatVertexAI = ChatVertexAI
sys.modules['langchain_community.chat_models.vertexai'] = _vmod

import langchain_community.llms as _ll
if not hasattr(_ll, 'VertexAI'):
    class VertexAI:  # placeholder
        pass
    _ll.VertexAI = VertexAI

# ---- 用 deepseek-v4-flash 作为评估 LLM（替代原版的 OpenAI gpt-3.5-turbo）----
from langchain_deepseek import ChatDeepSeek
from ragas import evaluate
from ragas.metrics import context_precision, context_recall

# 从项目根目录 .env 加载 DEEPSEEK_API_KEY（ChatDeepSeek 默认读 DEEPSEEK_API_KEY 环境变量）
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

LLM_NAME = "deepseek-v4-flash"
ragas_llm = ChatDeepSeek(model=LLM_NAME, temperature=1e-8)

# 嵌入模型：使用本地缓存的 bge-large-en-v1.5（位于 F:\Teewon\Milvue\models\hub）
from langchain_huggingface import HuggingFaceEmbeddings
EMB_NAME = "BAAI/bge-large-en-v1.5"
lc_embeddings = HuggingFaceEmbeddings(model_name=EMB_NAME)

# 评估数据集（llm / embeddings 直接传 LangChain 实例，ragas 自动包装）
ragas_result = evaluate(
    ragas_input_ds,
    metrics=[context_precision, context_recall],
    llm=ragas_llm,
    embeddings=lc_embeddings,
)

# 查看评估结果
ragas_output_df = ragas_result.to_pandas()
temp = ragas_output_df.fillna(0.0)
temp['context_f1'] = 2.0 * temp.context_precision * temp.context_recall \
                    / (temp.context_precision + temp.context_recall)
display(temp.head())

# 计算检索平均分
avg_retrieval_f1 = np.round(temp.context_f1.mean(), 2)
print(f"Using {eval_df.shape[0]} eval questions, Mean Retrieval F1 Score = {avg_retrieval_f1}")


C:\Users\Administrator\AppData\Local\Temp\ipykernel_31920\3900034482.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.llms as _ll
C:\Users\Administrator\AppData\Local\Temp\ipykernel_31920\3900034482.py:21: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import context_precision, context_recall
C:\Users\Administrator\AppData\Local\Temp\ipykernel_31920\3900034482.py:21: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import co

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/8 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,context_precision,context_recall,context_f1
0,What do the parameters for HNSW mean?,"[node closest to the target in this layer, and...",* `M`: Maximum number of outgoing connections ...,"* M: maximum degree, or number of connections ...",1.0,0.20,0.333333
1,What are good default values for HNSW paramete...,[Select your Milvus distribution first. Index ...,"* For `nprobe`, a reasonable default value is ...","M=16, efConstruction=32, ef=32",0.0,0.00,NaN
2,What does nlist vs nprobe mean in ivf_flat?,[FAQ What is the difference between FLAT index...,- `nlist` refers to the number of clusters (or...,# nlist: controls how the vector data is part...,1.0,0.75,0.857143
3,What is the default AUTOINDEX index and vector...,"[True, and auto_id is enabled for the primary ...",The default `AUTOINDEX` index type uses L2 (Eu...,Index type = HNSW and distance metric=IP Inner...,0.0,0.00,NaN


Using 4 eval questions, Mean Retrieval F1 Score = 0.6
